# 🍽️ Stage 1 — Data Parsing

**Goal:** Transform raw, unstructured recipe text into structured Python dicts.

This notebook covers:
- Loading the raw input data
- Parsing ingredients (quantity, unit, item)
- Extracting time and servings via regex
- Handling edge cases: fractions, ranges, inline sections, paragraph instructions
- Saving `parsed_data.json` for the next stage

---


## 0. Setup

In [ ]:
import sys, json, re, os

# Allow notebooks to import from the src package
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import pandas as pd
from src.parser import (
    extract_recipe_features,
    parse_ingredient,
    parse_quantity,
    extract_time_minutes,
    NLP_AVAILABLE,
)

RAW_FILE    = os.path.join(REPO_ROOT, 'mock_raw_data.json')
OUTPUT_FILE = os.path.join(REPO_ROOT, 'data', 'parsed_data.json')
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

print(f'spaCy NLP available: {NLP_AVAILABLE}')
print(f'Raw file: {RAW_FILE}')


## 1. Load Raw Data

In [ ]:
with open(RAW_FILE, 'r', encoding='utf-8') as f:
    raw_items = json.load(f)

print(f'Loaded {len(raw_items)} raw recipe entries.\n')
print('=== Sample raw entry ========================')
print(raw_items[0]['raw_text'])


## 2. Quantity Parsing — Fractions & Ranges

The parser converts fractions and ranges to floats:

In [ ]:
examples = ['3/4', '1/2', '9-11', '2', '2.5', None]
for ex in examples:
    print(f'  parse_quantity({str(ex)!r:10}) → {parse_quantity(ex)}')


## 3. Ingredient Line Parsing

Each ingredient line is broken into `quantity`, `unit`, `item`, and `normalized_item`:

In [ ]:
ingredient_samples = [
    '- 2 cups shredded chicken',
    '1.5 cups flour',
    '500 g potatoes, diced',
    '3/4 cup granulated sugar',
    '• 1 tbsp olive oil',
    'pinch of sea salt',
    'water',
    '200 g mozzarella cheese',
]

rows = [parse_ingredient(line) for line in ingredient_samples]
df = pd.DataFrame(rows)
df


## 4. Time Extraction

Three complementary regex patterns handle combined, hours-only, and minutes-only expressions:

In [ ]:
time_samples = [
    'prep time: 90 mins',
    'Takes 2 hours',
    'Time: 20 minutes',
    'Cooking time: 1 hour and 30 minutes',
    'Time takes: 10 minutes',
    'Bake for 15 min',
    'No time info here',
]
for t in time_samples:
    result = extract_time_minutes(t.lower())
    print(f'  {t!r:45} → {str(result)} min')


## 5. Parse All Recipes

In [ ]:
parsed_recipes = []
for item in raw_items:
    rid    = item.get('id', f'rec_{len(parsed_recipes)+1:03d}')
    text   = item.get('raw_text', '')
    recipe = extract_recipe_features(text, recipe_id=rid)
    parsed_recipes.append(recipe)
    n_ing  = len(recipe['ingredients'])
    n_step = len(recipe['instructions'])
    print(f"[{rid}] {recipe['name'][:45]:<45}  "
          f"{n_ing} ingredients  {n_step} steps  "
          f"time={recipe['prep_time_minutes']}  servings={recipe['servings']}")

print(f'\nTotal parsed: {len(parsed_recipes)} recipes')


## 6. Inspect a Parsed Recipe

In [ ]:
# Change index to inspect different recipes
idx = 0
r   = parsed_recipes[idx]

print(f'Recipe ID  : {r["recipe_id"]}')
print(f'Name       : {r["name"]}')
print(f'Time (min) : {r["prep_time_minutes"]}')
print(f'Servings   : {r["servings"]}')
print('\n--- Ingredients ---')
pd.DataFrame(r['ingredients'])


In [ ]:
print('--- Instructions ---')
for i, step in enumerate(r['instructions'], 1):
    print(f'  {i}. {step}')

print('\n--- Parse Warnings ---')
for w in r['_meta']['parse_warnings']:
    print(f'  ⚠ {w}')
if not r['_meta']['parse_warnings']:
    print('  ✓ No warnings')


## 7. Missing Values Summary (Pre-Imputation)

In [ ]:
summary = pd.DataFrame([{
    'recipe_id':         r['recipe_id'],
    'name':              r['name'],
    'n_ingredients':     len(r['ingredients']),
    'n_steps':           len(r['instructions']),
    'prep_time_minutes': r['prep_time_minutes'],
    'servings':          r['servings'],
    'has_warnings':      bool(r['_meta']['parse_warnings']),
} for r in parsed_recipes])

print('Missing value counts:')
print(summary.isnull().sum())
summary


## 8. Save Parsed Data

In [ ]:
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(parsed_recipes, f, indent=2, ensure_ascii=False)

print(f'✅ Saved {len(parsed_recipes)} parsed recipes → {OUTPUT_FILE}')
print('   Next step: run 02_enrichment.ipynb')
